In [ ]:
import os
import json
import copy
import cv2
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from scipy.integrate import odeint
from matplotlib.colors import LogNorm

def lorenz96(X, t, p):
    return np.roll(X, 1)*(np.roll(X, -1) - np.roll(X, 2)) - X + p

plt.rcParams["font.size"] = 18
results_dir = 'results'

In [ ]:
init = np.load(os.path.join(results_dir, 'init.npy'))

with open(os.path.join(results_dir, 'kwargs_sys.json')) as f:
    params = json.load(f)
dt = params['dt']
len_segs = params['len_segs']
num_segs = int(params['N_time_steps']/params['len_segs'])

X = np.array(pd.read_csv(os.path.join(results_dir, 'true.csv'), header=None)).T
estimated = np.zeros_like(X)
p = np.array(pd.read_csv(os.path.join(results_dir, 'p.csv'), header=None))
mse_true = np.array(pd.read_csv(os.path.join(results_dir, 'mse_true.csv'), header=None))
mse_measured = np.array(pd.read_csv(os.path.join(results_dir, 'mse_measured.csv'), header=None))

t = np.arange(0, len_segs*dt, dt)
mse_measured_min = np.argmin(mse_measured, axis=1)

for i in range(num_segs):
    idx = mse_measured_min[i]
    estimated[:, i*len_segs:(i+1)*len_segs] = odeint(lorenz96, init[i, :, idx], t, args=(p[i, idx],)).T

x = np.arange(0, (X.shape[1])*dt, dt)
y = np.arange(X.shape[0])


vmax = np.max([np.max(X), np.max(estimated)])
vmin = np.min([np.min(X), np.min(estimated)])
vmax_error = np.max(np.abs(estimated-X))
vmin_error = -vmax_error

x = cv2.resize(x, (1000, 9))
X = cv2.resize(X, (1000, 9))
estimated = cv2.resize(estimated, (1000, 9))

cmap = 'viridis'
cmap = mpl.colormaps.get_cmap('viridis')  # viridis is the default colormap for imshow
cmap.set_bad(color='k')


X_mask = copy.copy(X)
for i in range(3):
    X_mask[[1+3*i, 2+3*i], :] = np.nan

In [ ]:
fig = plt.figure(facecolor="white", figsize=(16, 12))

x = np.arange(0, (X.shape[1])*dt, dt)*10 #間引いたため時間方向のスケールが変わった
y = np.arange(X.shape[0])

ax1 = fig.add_subplot(3, 4, (1, 2))
mesh = ax1.pcolormesh(x, y, X, cmap=cmap, vmax=vmax, vmin=vmin)
ax1.set_xlabel(r'$t$')
ax1.set_ylabel(r'$k$')
ax1.text(-10, 8, '(a)')
ax1.set_yticks([0, 2, 4, 6, 8]) 
ax1.set_yticklabels([1, 3, 5, 7, 9])


ax2 = fig.add_subplot(3, 4, (3, 4))
X_mask = copy.copy(X)
for i in range(3):
    X_mask[[1+3*i, 2+3*i], :] = np.nan
ax2.pcolormesh(x, y, X_mask, cmap=cmap, vmax=vmax, vmin=vmin)
ax2.set_xlabel(r'$t$')
ax2.set_ylabel(r'$k$')
ax2.text(-10, 8, '(b)')
ax2.set_yticks([0, 2, 4, 6, 8]) 
ax2.set_yticklabels([1, 3, 5, 7, 9])


ax3 = fig.add_subplot(3, 4, (5, 6))
ax3.pcolormesh(x, y, estimated, cmap=cmap, vmax=vmax, vmin=vmin)
ax3.set_xlabel(r'$t$')
ax3.set_ylabel(r'$k$')
ax3.text(-10, 8, '(c)')
ax3.set_yticks([0, 2, 4, 6, 8]) 
ax3.set_yticklabels([1, 3, 5, 7, 9])


ax4 = fig.add_subplot(3, 4, (7, 8))
mesh_error = ax4.pcolormesh(x, y, (np.abs(estimated-X)), cmap='Blues', norm=LogNorm())
ax4.set_xlabel(r'$t$')
ax4.set_ylabel(r'$k$')
ax4.text(-10, 8, '(d)')
ax4.set_yticks([0, 2, 4, 6, 8]) 
ax4.set_yticklabels([1, 3, 5, 7, 9])


ax9 = fig.add_subplot(3, 4, (9, 12))
ax9.plot(p[np.arange(num_segs), mse_measured_min][3:], 'o')
ax9.hlines(8.17, xmin=0, xmax=25, linestyle='dashed', colors='k')
ax9.set_ylim([8, 8.4])
ax9.set_xlabel(r'segment number')
ax9.set_ylabel(r'$\hat{F}$')
ax9.text(-3, 8.45, '(e)')


ax5 = fig.add_subplot(3, 4, 2)
ax5.axis("off")
fig.colorbar(mesh, ax=ax5, anchor=(2, 0))

ax6 = fig.add_subplot(3, 4, 4)
ax6.axis("off")
fig.colorbar(mesh, ax=ax6, anchor=(2, 0))

ax7 = fig.add_subplot(3, 4, 6)
ax7.axis("off")
fig.colorbar(mesh, ax=ax7, anchor=(2, 0))

ax8 = fig.add_subplot(3, 4, 8)
ax8.axis("off")
fig.colorbar(mesh_error, ax=ax8, anchor=(2, 0))

plt.subplots_adjust(hspace=0.3, wspace=0.5)